# MCQA — task demo

Three sections:

1. **Causal model** — variables, mechanisms, sampled prompt.
2. **Templates & token positions** — how the choice symbols are located in the tokenized prompt, including the dynamic `correct_symbol` position.
3. **Counterfactual generators** — what changes between input and counterfactual under each generator.

Tokenization uses `gpt2` to keep the demo cheap and offline. No interventions are run — for that, see `analyses/locate/demo.ipynb`.

In [ ]:
from causalab.tasks.MCQA.causal_models import (
    positional_causal_model,
    NUM_CHOICES,
    OBJECTS,
    COLORS,
)
from causalab.tasks.MCQA.counterfactuals import (
    sample_answerable_question,
    different_symbol,
    same_symbol_different_position,
    random_counterfactual,
)
from causalab.tasks.MCQA.token_positions import create_token_positions

model = positional_causal_model
f"NUM_CHOICES={NUM_CHOICES}, |OBJECTS|={len(OBJECTS)}, |COLORS|={len(COLORS)}"

## 1. Causal model

Inputs: `template`, `object`, `color`, plus `choices[0]`/`choices[1]` (color labels) and `symbols[0]`/`symbols[1]` (letters). Computed: `raw_input`, `answer_position`, `answer`, `raw_output`. Sample one answerable question and inspect the trace.

In [ ]:
import random

random.seed(0)
trace = sample_answerable_question()  # already fully traced

print("Inputs:")
print(f"  object  = {trace['object']!r}")
print(f"  color   = {trace['color']!r}")
for i in range(NUM_CHOICES):
    print(
        f"  choices[{i}] = {trace[f'choices[{i}]']!r:<10}  symbols[{i}] = {trace[f'symbols[{i}]']!r}"
    )
print()
print("Computed:")
print(f"  answer_position = {trace['answer_position']}")
print(f"  answer          = {trace['answer']!r}")
print(f"  raw_output      = {trace['raw_output']!r}")
print()
print("raw_input:")
print(trace["raw_input"])

## 2. Templates & token positions

`create_token_positions(pipeline)` returns positions for each choice's symbol, the period after each symbol, the dynamic `correct_symbol` (and its period), and `last_token`. The dynamic positions read `color` / `choices[{i}]` from the input sample to pick the correct choice, so we don't need to know the answer ahead of time.

In [ ]:
from causalab.tasks.token_positions import LMPipeline

pipeline = LMPipeline("gpt2", max_new_tokens=1)
positions = create_token_positions(pipeline)
list(positions.keys())

In [ ]:
ids = pipeline.load([trace])["input_ids"][0].tolist()
decoded = [pipeline.tokenizer.decode([t]) for t in ids]
pad_id = pipeline.tokenizer.pad_token_id

print(f"correct symbol = {trace['answer']!r} at position {trace['answer_position']}")
print()
print(f"{'idx':>4}  {'token':<20}  positions")
for i, tok in enumerate(decoded):
    if ids[i] == pad_id:
        continue
    hits = [name for name, pos in positions.items() if i in pos.index(trace)]
    marker = ", ".join(hits) if hits else ""
    print(f"{i:>4}  {tok!r:<20}  {marker}")

Note that `correct_symbol` lands on the same index as one of `symbols[0]` / `symbols[1]` — whichever corresponds to the matching color. Re-sample to confirm it tracks the answer:

In [ ]:
for _ in range(3):
    t = sample_answerable_question()
    correct_idx = positions["correct_symbol"].index(t)[0]
    sym0_idx = positions["symbols[0]"].index(t)[0]
    sym1_idx = positions["symbols[1]"].index(t)[0]
    print(
        f"answer_position={t['answer_position']}  answer={t['answer']!r}  "
        f"correct_symbol@{correct_idx}  symbols[0]@{sym0_idx}  symbols[1]@{sym1_idx}"
    )

## 3. Counterfactual generators

MCQA ships with three generators. The default for `generate_dataset` is `different_symbol` because it deconfounds `answer_position` (the typical target variable) from `answer` (the surface token). Each call below produces one `{input, counterfactual_inputs}` pair; we render both side-by-side.

In [ ]:
def show_pair(label, ex):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    print(f"--- {label} ---")
    print(
        f"  base : color={base['color']:<8} symbols=({base['symbols[0]']}, {base['symbols[1]']}) "
        f"choices=({base['choices[0]']}, {base['choices[1]']})  -> answer_position={base['answer_position']}, answer={base['answer']!r}"
    )
    print(
        f"  cf   : color={cf['color']:<8} symbols=({cf['symbols[0]']}, {cf['symbols[1]']}) "
        f"choices=({cf['choices[0]']}, {cf['choices[1]']})  -> answer_position={cf['answer_position']}, answer={cf['answer']!r}"
    )
    print()


random.seed(1)
show_pair("different_symbol", different_symbol())
show_pair("same_symbol_different_position", same_symbol_different_position())
show_pair("random_counterfactual", random_counterfactual())

Reading the output:

- **`different_symbol`** — `answer_position` is identical, `answer` letter changes. Use this when the analysis targets `answer_position`.
- **`same_symbol_different_position`** — symbol set is identical, `answer_position` flips. Use this when targeting `answer` (token identity).
- **`random_counterfactual`** — every input variable may differ. Useful for distribution-level baselines.

## Next steps

The onboarding tutorial runs its experiments on this task. Its locate scan, [06_localize](../../../demos/onboarding_tutorial/06_localize.md), runs from the repository root with:

```bash
uv run causalab run demos/onboarding_tutorial/workflows/mcqa_locate.json \
    --engine auto \
    --data-root demos/onboarding_tutorial/artifacts/data \
    --out runs/mcqa_locate \
    --device cuda
```

The run writes its outputs under `--out`. [Running experiments](../../../docs/running_experiments.md) shows how to write, validate and run a document of your own.